# Openskill Ratings

Calculate ratings using openskill algorithm for a given season. This file needs to be ran for every relevant season (starting in 2012).

In [1]:
import pandas as pd

season = 2025

df = pd.read_parquet(fr'..\data\unprocessed\womens_sports_reference\full_season_sports_reference_{season}.parquet')

df = df.loc[df['NCAA Tournament'] == 0, :].reset_index(drop=True)  # only include games before NCAA Tournament

df

,Team,Date,NCAA Tournament,Location,Opponent,Type,Result,Team Score,Opponent Score,Team FG,...,Opponent TOV,Opponent PF,Overtimes Amount,Overtime,Score Differential,Adjusted Score Differential,Possessions,Team PPP,Opponent PPP,Tempo
0,Abilene Christian,2024-11-08,0,-1,Florida International,REG (Non-Conf),1,76.0,73.0,21.0,...,18.0,32.0,0,0,3.0,1.166625,82.52,0.920989,0.884634,82.52
1,Abilene Christian,2024-11-12,0,1,Stephen F. Austin,REG (Non-Conf),-1,65.0,68.0,25.0,...,22.0,22.0,0,0,-3.0,1.166625,74.42,0.873421,0.913733,74.42
2,Abilene Christian,2024-11-16,0,1,North Texas,REG (Non-Conf),-1,60.0,71.0,24.0,...,23.0,17.0,0,0,-11.0,1.399871,76.88,0.780437,0.923517,76.88
3,Abilene Christian,2024-11-19,0,-1,Texas Tech,REG (Non-Conf),-1,40.0,66.0,12.0,...,26.0,20.0,0,0,-26.0,1.500000,67.42,0.593296,0.978938,67.42
4,Abilene Christian,2024-11-21,0,1,Western Kentucky,REG (Non-Conf),-1,64.0,68.0,23.0,...,17.0,16.0,0,0,-4.0,1.214668,72.10,0.887656,0.943135,72.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10883,Youngstown State,2025-02-20,0,-1,Detroit Mercy,REG (Conf),1,67.0,59.0,26.0,...,14.0,21.0,0,0,8.0,1.338710,73.50,0.911565,0.802721,73.50
10884,Youngstown State,2025-02-22,0,-1,Oakland,REG (Conf),1,52.0,51.0,19.0,...,13.0,19.0,0,0,1.0,1.000000,67.78,0.767188,0.752434,67.78
10885,Youngstown State,2025-02-26,0,-1,Robert Morris,REG (Conf),-1,53.0,76.0,18.0,...,13.0,15.0,0,0,-23.0,1.500000,68.44,0.774401,1.110462,68.44
10886,Youngstown State,2025-03-01,0,1,Cleveland State,REG (Conf),1,73.0,70.0,24.0,...,11.0,21.0,0,0,3.0,1.166625,66.24,1.102053,1.056763,66.24


Filter to just wins because I don't need duplicate perspectives

In [2]:
df = df.loc[
    df['Result'] == 1, 
    ['Date', 'Team', 'Location', 'Opponent', 'Result', 'Adjusted Score Differential']
].reset_index(drop=True)

df.sort_values(['Date'], inplace=True, ignore_index=True)

df

,Date,Team,Location,Opponent,Result,Adjusted Score Differential
0,2024-11-04,Kentucky,1,South Carolina Upstate,1,1.500000
1,2024-11-04,James Madison,-1,Kent State,1,1.399871
2,2024-11-04,North Carolina,1,Charleston Southern,1,1.500000
3,2024-11-04,Charlotte,1,Presbyterian,1,1.448039
4,2024-11-04,Quinnipiac,-1,Holy Cross,1,1.214668
...,...,...,...,...,...,...
5439,2025-03-15,Fairfield,0,Quinnipiac,1,1.500000
5440,2025-03-16,Lehigh,1,Army,1,1.417062
5441,2025-03-16,William & Mary,0,Campbell,1,1.166625
5442,2025-03-16,Murray State,0,Belmont,1,1.500000


In [3]:
def rescale_weights(arr, minimum, maximum):
    """
    Rescale the weights array to match desired weights.
    Assumes the data is already transformed where a 1 point win is 1.00, a blowout is 1.50, and a tie is 0.50.
    Ties will be weighted as half the minimum.
    """

    arr_scaled = ((arr - 1.00) / 0.50) * (maximum - minimum) + minimum
    arr_scaled[arr == 0.50] = minimum / 2
    return arr_scaled

In [4]:
from typing import Tuple
from openskill.models import BradleyTerryFull
import numpy as np

def get_season_ratings(X_train_weights: np.array, sigma: float, hfa_mu: float) -> Tuple[BradleyTerryFull, dict]:
    """
    Run the algorithm over the course of a season.

    Args:
        X_train_weights (np.array): Numpy array where each row is a game and columns are Winner, Loser, Location, Weight.
        sigma (float): The starting variance for each team rating.
        hfa_mu (float): The mu of home field advantage.

    Returns:
        Tuple[BradleyTerryFull, dict]: The environment object and dictionary of team ratings.
    """
    # initialize
    env = BradleyTerryFull(sigma=sigma)

    teams = set(X_train_weights[:, 0]).union(set(X_train_weights[:, 1]))
    team_ratings = dict(zip(teams, [env.rating() for _ in range(len(teams))]))
    hfa = env.rating(mu=hfa_mu, sigma=0.0)

    # iterate through games
    for winner, loser, location, weight in X_train_weights:
        winner_rating = team_ratings[winner]
        loser_rating = team_ratings[loser]
        if location == 1:
            [[winner_post, _], [loser_post]] = env.rate([[winner_rating, hfa], [loser_rating]])
        elif location == -1:
            [[winner_post], [loser_post, _]] = env.rate([[winner_rating], [loser_rating, hfa]])
        else:
            [[winner_post], [loser_post]] = env.rate([[winner_rating], [loser_rating]])

        winner_mu_adjustment = (winner_post.mu - winner_rating.mu)*weight
        winner_sigma_adjustment = (winner_post.sigma - winner_rating.sigma)*weight
        team_ratings[winner] = env.rating(mu=winner_rating.mu + winner_mu_adjustment, sigma=winner_rating.sigma + winner_sigma_adjustment)

        loser_mu_adjustment = (loser_post.mu - loser_rating.mu)*weight
        loser_sigma_adjustment = (loser_post.sigma - loser_rating.sigma)*weight
        team_ratings[loser] = env.rating(mu=loser_rating.mu + loser_mu_adjustment, sigma=loser_rating.sigma + loser_sigma_adjustment)

    return env, team_ratings

In [5]:
hfa_mu = 0.50
sigma = 25/3
minimum = 2/3
maximum = 4/3


X_train = df[['Team', 'Opponent', 'Location']].to_numpy()
weights = rescale_weights(df['Adjusted Score Differential'].to_numpy(), minimum, maximum)

X_train_weights = np.column_stack((X_train, weights))

env, team_ratings = get_season_ratings(X_train_weights, sigma, hfa_mu)

len(team_ratings)

362

In [6]:
df_ratings = pd.DataFrame(
    {
        'Team': team_ratings.keys(),
        'Mu': [v.mu for v in team_ratings.values()],
        'Sigma': [v.sigma for v in team_ratings.values()],
    }
)

df_ratings['OS Rating'] = df_ratings['Mu'] - df_ratings['Sigma']*3

df_ratings.sort_values(['OS Rating', 'Mu'], ascending=False, ignore_index=True, inplace=True)

df_ratings.head(50)

,Team,Mu,Sigma,OS Rating
0,South Carolina,57.154437,4.074368,44.931333
1,UCLA,56.918023,4.208835,44.291517
2,Texas,55.779370,4.133418,43.379118
3,Southern California,55.653531,4.230876,42.960902
4,Connecticut,54.559248,4.185392,42.003074
5,TCU,53.152675,4.196784,40.562322
6,Duke,50.381608,3.956370,38.512498
7,Notre Dame,49.969382,4.205796,37.351993
8,NC State,49.392994,4.118980,37.036053
9,Baylor,48.861583,4.003313,36.851645


In [7]:
df_ratings.to_parquet(f'../data/preprocessed/womens_os_rankings/os_rankings_{season}.parquet')

'Done'

'Done'